# V4.8: masked-mean humanized-AI experiment

Run cells from top to bottom, one at a time, on a Colab T4 GPU. V4.8 leaves V4.0 unchanged and never reads GRADTEX or the sealed RAID-derived cohort.

**Order:** setup → unused PADBen diagnostic → masked-mean training → diagnostics → decide whether the optional optimiser run is justified → calibrate one selected model.

## 1. Check GPU

A T4 GPU is sufficient. If CUDA is false, use **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())

## 2. Get the V4.8 code

This pulls the dedicated V4.8 training, diagnostics, and calibration code.

In [ ]:
!git clone https://github.com/bonbon1235312/googlecolab-humanize-detector.git /content/humanized-ai-likelihood || true
%cd /content/humanized-ai-likelihood
!git pull -q
%cd /content/humanized-ai-likelihood/ml
!pip install -q -e .

## 3. Mount Drive and set paths

`control-v1` is the V4 control data you already created. This notebook does not rebuild it.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
CONTROL_DATA_DIR = Path('/content/drive/MyDrive/v4-data/control-v1')
PADBEN_DIAGNOSTIC_DIR = Path('/content/drive/MyDrive/v4-data/padben-unused-diagnostic-v1')
V48_ROOT = Path('/content/drive/MyDrive/v4-artifacts/v4-8')

print('Control data exists:', CONTROL_DATA_DIR.exists())
print('Train file exists:', (CONTROL_DATA_DIR / 'train.jsonl').exists())

## 4. Create the unused PADBen diagnostic — run once

This creates a separate 1,000-human / 1,000-AI in-source diagnostic from PADBen rows not present in V4 control data. It is never used for training or calibration.

In [ ]:
from datasets import load_dataset
from humanized_detector.v4_control_data import prepare_padben_diagnostic

padben_rows = [dict(row) for row in load_dataset('JonathanZha/PADBen', 'exhaustive-task5', split='train')]
report = prepare_padben_diagnostic(
    padben_rows, CONTROL_DATA_DIR, PADBEN_DIAGNOSTIC_DIR, samples_per_class=1000
)
print(report)

## 5. Train V4.8 masked-mean

This is the main run. It changes only the encoder's token pooling from V4.0's first token to a padding-safe mean; data, model capacity, seed, and main optimiser settings remain comparable.

In [ ]:
!python -u -m humanized_detector.v4_train \
  --data-dir /content/drive/MyDrive/v4-data/control-v1 \
  --artifacts-dir /content/drive/MyDrive/v4-artifacts/v4-8/masked_mean_base \
  --capacity 5m \
  --token-pooling masked_mean \
  --epochs 6 \
  --batch-size 64 \
  --lr 3e-5 \
  --weight-decay 0.01

## 6. Diagnose the finished checkpoint

This reads train, development, and the separate PADBen diagnostic only. It does **not** read calibration, GRADTEX, or RAID.

In [ ]:
from humanized_detector.v4_diagnostics import score_padben_diagnostic, write_v4_fit_diagnostics

ARTIFACTS_DIR = V48_ROOT / 'masked_mean_base'
fit = write_v4_fit_diagnostics(CONTROL_DATA_DIR, ARTIFACTS_DIR, bootstrap_iterations=1000)
padben = score_padben_diagnostic(PADBEN_DIAGNOSTIC_DIR, ARTIFACTS_DIR)

print('Train/development AUC gap:', fit['train_development_roc_auc_gap'])
print('Development subtype results:', fit['development']['subtypes'])
print('Unused PADBen result:', padben)

## 7. Optional: one prespecified optimiser run

Only run this after recording the masked-mean diagnostics and deciding it is warranted. It uses `1e-4` learning rate, 400-step warm-up, gradient clipping, and lower label smoothing. It is not an open-ended sweep.

In [ ]:
# Run this only after discussing the diagnostic output.
# !python -u -m humanized_detector.v4_train \
#   --data-dir /content/drive/MyDrive/v4-data/control-v1 \
#   --artifacts-dir /content/drive/MyDrive/v4-artifacts/v4-8/masked_mean_optimised \
#   --capacity 5m --token-pooling masked_mean --epochs 6 --batch-size 64 \
#   --lr 1e-4 --weight-decay 0.01 --label-smoothing 0.02 \
#   --warmup-steps 400 --grad-clip-norm 1.0

## 8. Calibrate one selected model

Run this only after selecting one candidate from development results. The report includes prompt-lineage-safe cross-fitted Platt diagnostics plus 1%, 2%, and 5% human-FPR operating points.

In [ ]:
# Leave this as masked_mean_base unless a later development-only decision selects the optimiser run.
SELECTED_ARTIFACTS = V48_ROOT / 'masked_mean_base'
!python -u -m humanized_detector.v4_calibrate --data-dir $CONTROL_DATA_DIR --artifacts-dir $SELECTED_ARTIFACTS

## Do not run final evaluation yet

Do not inspect or evaluate the sealed RAID-derived cohort until model selection and calibration are frozen. GRADTEX remains a regression benchmark, not a tuning target.